<a href="https://colab.research.google.com/github/Kushagraraghav/projects/blob/main/Steganography.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pillow numpy


In [10]:
from google.colab import files
uploaded = files.upload()




Saving example_input.png to example_input.png
Saving example_input.wav to example_input.wav


In [11]:
import struct
from PIL import Image, PngImagePlugin
import numpy as np
import wave

# ---------- Utilities ----------
def _to_bits(data_bytes):
    for b in data_bytes:
        for i in range(7, -1, -1):
            yield (b >> i) & 1

def _bits_to_bytes(bits):
    b = 0
    out = bytearray()
    cnt = 0
    for bit in bits:
        b = (b << 1) | bit
        cnt += 1
        if cnt == 8:
            out.append(b)
            b = 0
            cnt = 0
    return bytes(out)

def _int_to_32bits(n):
    return [(n >> i) & 1 for i in range(31, -1, -1)]

def _bits_to_int(bits):
    n = 0
    for bit in bits:
        n = (n << 1) | bit
    return n


# ---------- FIXED Image LSB ----------
class ImageLSB:
    @staticmethod
    def embed(input_image_path, output_image_path, message):
        img = Image.open(input_image_path).convert('RGBA')
        arr = np.array(img).astype(np.uint8)   # <-- FIXED
        flat = arr.flatten()

        data = message.encode('utf-8')
        length = len(data)

        header_bits = _int_to_32bits(length)
        data_bits = list(_to_bits(data))
        bits = header_bits + data_bits

        if len(bits) > len(flat):
            raise ValueError("Message too large for this image.")

        flat_copy = flat.copy()

        for i, bit in enumerate(bits):
            flat_copy[i] = (flat_copy[i] & 254) | bit   # <-- FIXED

        new_arr = flat_copy.reshape(arr.shape)
        Image.fromarray(new_arr.astype('uint8')).save(output_image_path)
        print("Image Stego Created:", output_image_path)

    @staticmethod
    def extract(stego_image_path):
        img = Image.open(stego_image_path).convert('RGBA')
        flat = np.array(img).astype(np.uint8).flatten()   # <-- FIXED

        header_bits = [flat[i] & 1 for i in range(32)]
        length = _bits_to_int(header_bits)

        data_bits = [flat[i] & 1 for i in range(32, 32 + length * 8)]
        return _bits_to_bytes(data_bits).decode()


# ---------- Audio LSB ----------
class AudioLSB:
    @staticmethod
    def embed(input_wav_path, output_wav_path, message):
        with wave.open(input_wav_path, 'rb') as wf:
            params = wf.getparams()
            frames = bytearray(wf.readframes(wf.getnframes()))

        data = message.encode()
        length = len(data)

        header_bits = _int_to_32bits(length)
        data_bits = list(_to_bits(data))
        bits = header_bits + data_bits

        if len(bits) > len(frames):
            raise ValueError("Message too large for audio.")

        frames_mod = frames[:]

        for i, bit in enumerate(bits):
            frames_mod[i] = (frames_mod[i] & 254) | bit

        with wave.open(output_wav_path, 'wb') as wf:
            wf.setparams(params)
            wf.writeframes(bytes(frames_mod))

        print("Audio Stego Created:", output_wav_path)

    @staticmethod
    def extract(stego_path):
        with wave.open(stego_path, 'rb') as wf:
            frames = bytearray(wf.readframes(wf.getnframes()))

        length = _bits_to_int([frames[i] & 1 for i in range(32)])
        data_bits = [frames[32 + i] & 1 for i in range(length * 8)]

        return _bits_to_bytes(data_bits).decode()


# ---------- Zero Width Steganography ----------
class TextZWC:
    Z0 = '\u200b'
    Z1 = '\u200c'
    END = '\u200d'

    @staticmethod
    def encode_message(msg):
        bits = ''.join(format(b, '08b') for b in msg.encode())
        encoded = ''.join(TextZWC.Z0 if bit == '0' else TextZWC.Z1 for bit in bits)
        return encoded + TextZWC.END

    @staticmethod
    def decode_message(encoded):
        bits = ""
        for ch in encoded:
            if ch == TextZWC.Z0:
                bits += '0'
            elif ch == TextZWC.Z1:
                bits += '1'
            elif ch == TextZWC.END:
                break

        decoded = bytearray()
        for i in range(0, len(bits), 8):
            decoded.append(int(bits[i:i+8], 2))

        return decoded.decode()

    @staticmethod
    def embed(host_text, secret):
        return host_text + TextZWC.encode_message(secret)

    @staticmethod
    def extract(stego_text):
        zwc_chars = ''.join(ch for ch in stego_text if ch in [TextZWC.Z0, TextZWC.Z1, TextZWC.END])
        return TextZWC.decode_message(zwc_chars)


# ---------- PNG Metadata ----------
class PngMetadata:
    @staticmethod
    def embed(input_png, output_png, key, message):
        img = Image.open(input_png)
        meta = PngImagePlugin.PngInfo()
        meta.add_text(key, message)
        img.save(output_png, pnginfo=meta)
        print("PNG Metadata Stego Created:", output_png)

    @staticmethod
    def extract(png_file, key):
        img = Image.open(png_file)
        return img.info.get(key, "")


In [12]:
from google.colab import files

# Embed secret message into the image
ImageLSB.embed("example_input.png", "stego_image.png", "Hello from Colab!")

# Extract message to verify
print("Extracted:", ImageLSB.extract("stego_image.png"))

# Download output image
files.download("stego_image.png")



Image Stego Created: stego_image.png
Extracted: Hello from Colab!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
from google.colab import files

# Embed secret text into WAV audio
AudioLSB.embed("example_input.wav", "stego_audio.wav", "Secret Audio Message from Colab")

# Extract message to verify
print("Extracted:", AudioLSB.extract("stego_audio.wav"))

# Download stego audio
files.download("stego_audio.wav")




Audio Stego Created: stego_audio.wav
Extracted: Secret Audio Message from Colab


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
host = "This is visible classroom text."
stego = TextZWC.embed(host, "HiddenText123")

print("Extracted:", TextZWC.extract(stego))


Extracted: HiddenText123


In [15]:
from google.colab import files

# Hide message inside PNG metadata
PngMetadata.embed("example_input.png", "metadata_stego.png", "secret-key", "Metadata Hidden Text in PNG")

# Extract message
print("Extracted:", PngMetadata.extract("metadata_stego.png", "secret-key"))

# Download metadata stego file
files.download("metadata_stego.png")


PNG Metadata Stego Created: metadata_stego.png
Extracted: Metadata Hidden Text in PNG


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import os

print("Original Image Size:", os.path.getsize("example_input.png"), "bytes")
print("Stego Image Size   :", os.path.getsize("stego_image.png"), "bytes")


Original Image Size: 74007 bytes
Stego Image Size   : 84357 bytes


In [17]:
hidden = ImageLSB.extract("stego_image.png")
print("Hidden Message Found Inside Image:", hidden)


Hidden Message Found Inside Image: Hello from Colab!
